## MVP ENGENHARIA DE DADOS
Aluna: Bianca Goncalves de Oliveira

Matricula: 4052025002425

### Contexto de Negócios e Perguntas 

Este trabalho tem como objetivo construir um modelo de dados por meio de pipelines de ETL desenvolvidos em SQL e PySpark, seguindo a arquitetura em camadas Bronze, Silver e Gold. A camada Bronze concentra a ingestão dos dados brutos tal como recebidos da fonte; a camada Silver reúne os processos de tratamento, padronização e verificação da qualidade dos dados; e a camada Gold disponibiliza um conjunto de dados consolidado e pronto para consumo analítico, voltado a responder às perguntas de negócio definidas para o projeto.

O projeto foi orientado pelas seguintes perguntas analíticas, relacionadas aos gêneros cinematográficos e à sua evolução ao longo do tempo:
- Qual é o ranking dos gêneros de filmes lançados em um determinado ano, considerando a média das avaliações recebidas?
- Quais são os 10 filmes (ou combinações de ano e gênero, na granularidade da tabela final) com as maiores notas, e quais os respectivos anos de lançamento e gêneros?
- Dado um gênero específico, quais são os anos com as melhores avaliações médias, entre as 50 melhores combinações?
- Quais são os anos que concentram o maior número de votos registrados, independentemente do gênero?

A base utilizada é a disponibilizada publicamente pelo IMDb (Internet Movie Database), que reúne informações sobre filmes, séries e documentários. Do conjunto completo de arquivos oferecidos pelo IMDb, foram importadas duas tabelas para este projeto: uma com dados descritivos dos títulos e outra com as métricas de avaliação atribuídas pelos usuários. Cada arquivo é distribuído em formato TSV (valores separados por tabulação), compactado em gzip, codificado em UTF-8, com cabeçalho na primeira linha e o marcador '\N' para indicar valores ausentes ou nulos.

### Carga dos Dados

A carga dos dados foi realizada em um notebook Databricks, utilizando uma SparkSession já provisionada pelo ambiente. Antes da ingestão, o banco de dados da camada Bronze foi removido (caso já existisse) e recriado, garantindo um ambiente limpo e evitando conflitos com tabelas homônimas de execuções anteriores.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
spark.sql("DROP DATABASE IF EXISTS bronze CASCADE")

In [0]:
spark.sql("CREATE DATABASE bronze")

Os dois arquivos TSV compactados (title.basics.tsv.gz e title.ratings.tsv.gz), previamente enviados ao workspace do Databricks, foram lidos com spark.read, configurando o separador de tabulação, a descompactação gzip, o cabeçalho como nome das colunas e o marcador '\N' como valor nulo. Durante a leitura de title.basics, a coluna isAdult apresentou erro em sua criação e foi descartada diretamente no carregamento, já que essa informação não seria utilizada para responder às perguntas da análise.

In [0]:
title_basics_spark_df = (spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("compression", "gzip")
    .option("nullValue", "\\N")
    .csv("/Workspace/Users/oliveirabianca@id.uff.br/DbAcademy/title.basics.tsv.gz")
    .drop("isAdult"))

In [0]:

title_ratings_spark_df = (spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("compression", "gzip")
    .option("nullValue", "\\N")
    .csv("/Workspace/Users/oliveirabianca@id.uff.br/DbAcademy/title.ratings.tsv.gz"))

Para tornar a escrita das tabelas mais resiliente a quedas de cluster por inatividade, foi criada uma função auxiliar (run_with_retry) que reexecuta a operação de escrita uma vez, recriando a sessão Spark, caso o erro capturado seja do tipo INACTIVITY_TIMEOUT.

In [0]:
def run_with_retry(func, max_retries=1):
    for attempt in range(max_retries + 1):
        try:
            return func()
        except Exception as e:
            if "INACTIVITY_TIMEOUT" in str(e) and attempt < max_retries:
                global spark
                spark = DatabricksSession.builder.getOrCreate()
                continue
            raise

Os DataFrames foram entao persistidos como tabelas gerenciadas na camada bronze:

In [0]:
run_with_retry(lambda: title_basics_spark_df.write.mode("overwrite").saveAsTable("bronze.title_basics"))

In [0]:
run_with_retry(lambda: title_ratings_spark_df.write.mode("overwrite").saveAsTable("bronze.title_ratings"))

Após a carga, o conteúdo das tabelas foi inspecionado por meio de consultas SQL simples, confirmando o sucesso da ingestão. Essa inspeção também permitiu identificar, na coluna titleType, os valores distintos de tipo de título disponíveis; como o projeto está delimitado à análise de produções cinematográficas, apenas os tipos movie, tvShort, short e tvMovie foram mantidos nas etapas seguintes.

In [0]:
%sql
SELECT * FROM bronze.title_basics LIMIT 20

In [0]:
%sql
SELECT DISTINCT titleType FROM bronze.title_basics;

In [0]:
%sql
SELECT * FROM bronze.title_ratings LIMIT 20

### Pipeline de Dados

Todo o pipeline de ETL foi implementado em um único notebook do Databricks, organizado sequencialmente em três blocos correspondentes às camadas Bronze, Silver e Gold, sem ramificação em notebooks separados. Dentro de cada bloco, células PySpark (leitura, transformação e escrita dos DataFrames) foram intercaladas com células %sql, usadas para criar/recriar os bancos de dados, validar a qualidade dos dados e materializar as tabelas finais. A função run_with_retry, apresentada na seção de carga, foi reaproveitada nas escritas realizadas via PySpark para tornar o notebook mais tolerante a interrupções do cluster. 

O banco de dados da camada Silver foi recriado e a tabela de avaliações foi copiada diretamente da camada Bronze:

In [0]:
%sql
DROP DATABASE silver CASCADE

In [0]:
%sql
CREATE DATABASE silver

In [0]:
%sql
CREATE TABLE silver.title_ratings AS
SELECT * FROM bronze.title_ratings;

Em seguida, a tabela de títulos foi transformada com a função explode sobre a coluna de gêneros (originalmente com múltiplos valores separados por vírgula), de modo que cada gênero de um título passasse a ocupar uma linha própria. Nesse mesmo passo, colunas consideradas irrelevantes para os objetivos do projeto foram descartadas.

In [0]:
from pyspark.sql.functions import explode, split, col

title_genres_exploded_df = title_basics_spark_df.select(
    col("tconst"),
    col("primaryTitle"),
    col("titleType"),
    col("startYear"),
    explode(split(col("genres"), ",")).alias("genre")
)

title_genres_exploded_df.show(10)

In [0]:
title_genres_exploded_df.write.mode("overwrite").saveAsTable("silver.title_basics")

In [0]:
%sql
SELECT * FROM silver.title_basics LIMIT 100

### Qualidade de Dados

Ao longo da camada Silver, cada coluna relevante foi verificada quanto à presença de nulos, valores vazios ou fora do domínio esperado, e os problemas identificados foram tratados como descrito a seguir.

In [0]:
%sql
SELECT COUNT(*) 
FROM silver.title_basics 
WHERE tconst IS NULL;

Não foram encontrados valores nulos em tconst.

In [0]:
%sql
SELECT *
FROM silver.title_basics
WHERE primaryTitle IS NULL OR TRIM(primaryTitle) = '' OR primaryTitle = '\N'

Foram identificadas inconsistências em primaryTitle (nulos, vazios ou marcados como '\N'); optou-se por substituir esses valores por 'Unknown', preservando a linha em vez de descartá-la:

In [0]:
%sql
UPDATE silver.title_basics
SET primaryTitle = 'Unknown'
WHERE (primaryTitle IS NULL OR primaryTitle = '\N' OR TRIM(primaryTitle) = '')

In [0]:

%sql
SELECT COUNT(*) 
FROM silver.title_basics
WHERE titleType IS NULL OR TRIM(titleType) = '' OR primaryTitle = '\N';

Não foram encontrados valores nulos em titleType.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics 
WHERE startYear IS NULL OR startyear = '\N';

In [0]:
%sql
DELETE FROM silver.title_basics 
WHERE startYear IS NULL OR startyear = '\N';
     

Como startYear é essencial para a análise, as linhas com esse campo nulo foram removidas da base.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics 
WHERE startYear = '';
     

Como os valores de startYear estavam armazenados como string, sendo necessaria a conversao para INT a fim de possibilitar comparacoes e agragacoes numéricas. Para isso, vrou-se a oluna startYear_int.  

In [0]:
%sql
ALTER TABLE silver.title_basics 
ADD COLUMN startYear_int INT;

UPDATE silver.title_basics 
SET startYear_int = CAST(startYear AS INT);
     

In [0]:
%sql
SELECT MIN(startYear_int), MAX(startYear_int) 
FROM silver.title_basics

Como o IMDB também cataloga títulos ainda nao lancados, optou-se por excluir os registros com startYear igual ou posterior a 2026. 

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics
WHERE startYear_int > 2025

In [0]:
%sql
DELETE FROM silver.title_basics
WHERE startYear_int > 2025;

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics
WHERE startYear_int > 2025

Como o gênero é uma variável essencial para as análises propostas, os registros sem essa informação foram removidos da base. Uma consulta de distribuição por gênero confirmou volume expressivo de registros em cada categoria remanescente.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_basics
WHERE genre IS NULL OR genre = '/N' OR TRIM(genre) = '';

In [0]:
%sql
SELECT genre, COUNT(*)
FROM silver.title_basics
GROUP BY genre
ORDER BY COUNT(*) DESC

Concluídas as verificações na tabela de títulos, foram realizadas validações equivalentes na tabela de avaliações. Não foram encontrados valores nulos em tconst, averageRating ou numVotes.

In [0]:
%sql
SHOW TABLES IN silver;

In [0]:
%sql
SELECT *
FROM silver.title_ratings LIMIT 50;

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_ratings
WHERE tconst IS NULL OR TRIM(tconst) = '' OR tconst = '\\N';

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_ratings
WHERE averageRating IS NULL OR TRIM(averageRating) = '';

In [0]:
%sql
SELECT COUNT (*)
FROM silver.title_ratings
WHERE numVotes IS NULL OR TRIM(numVotes) = '';

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_ratings
WHERE CAST(averageRating AS DOUBLE) < 0 OR CAST(averageRating AS DOUBLE) > 10;

Também não foram encontradas notas fora do intervalo válido de 0 a 10, nem títulos com zero votos registrados.

In [0]:
%sql
SELECT count(*) 
FROM silver.title_ratings
WHERE numVotes = 0;

Nao há filmes com 0 numeros de votos.

In [0]:
%sql
SELECT COUNT(*)
FROM silver.title_ratings r
LEFT JOIN silver.title_basics b
ON r.tconst = b.tconst
WHERE b.tconst IS NULL;

A correspondência entre as tabelas foi verificada por meio de uma junção (LEFT JOIN) de title_ratings com title_basics, checando a existência de tconst sem correspondente na tabela de títulos.

In [0]:
%sql
CREATE TABLE silver.title_joined AS
SELECT
  b.tconst,
  b.titleType,
  b.primaryTitle,
  b.startYear,
  b.genre,
  r.averageRating,
  r.numVotes
FROM silver.title_ratings r
LEFT JOIN silver.title_basics b
ON r.tconst = b.tconst;

In [0]:
%sql
SELECT *
FROM silver.title_joined LIMIT 100;

Após a conclusao das atividades prevista para a camada prata, iniciou-se a criacao da camada ouro.

In [0]:
%sql
DROP DATABASE IF EXISTS gold CASCADE;

In [0]:
%sql
CREATE DATABASE gold;

In [0]:
%sql
CREATE TABLE gold.genre_analysis AS
SELECT
    startYear AS year,
    genre,
    AVG(CAST(averageRating AS DOUBLE)) AS avgRating,
    SUM(CAST(numVotes AS INT)) AS totalVotes 
FROM
    silver.title_joined
WHERE
    titleType IN ('movie', 'tvShort', 'short', 'tvMovie')
GROUP BY
    startYear, 
    genre
ORDER BY
    startYear,
    genre;

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
WHERE year > 1980 LIMIT 500;

In [0]:
%sql
UPDATE gold.genre_analysis
SET avgRating = ROUND(avgRating, 3);

Por fim, já na camada Gold, os valores de avgRating foram arredondados para três casas decimais, padronizando a apresentação dos resultados.

### Análise de Dados

Com a tabla gold.genre_analysis consolidadem foram executadas as consultas necessárias para responder as perguntas de negócio definidas na seção 1.

1. Ranking de gênero por ano, segundo a média de avaliações

In [0]:
%sql
SELECT DISTINCT *
FROM gold.genre_analysis
WHERE year = 1980 
ORDER BY avgRating DESC;

O gênero Documentary apresentou maior avaliação média em 1980, enquanto Horor teve o pior desempenho entre os gêneros considerados. Repetindo a mesma consulta para 1990:

In [0]:
%sql
SELECT DISTINCT *
FROM gold.genre_analysis
WHERE year = 1990 
ORDER BY avgRating DESC;

Dez anos depois, Documentary perde a primeira posição para Reality-TV; esse resultado deve ser interpretado com cautela, pois esse gênero está associado a um número de votos bastante reduzido e, considerando que a tabela final foi construída apenas com títulos classificados como filmes, sua presença pode indicar alguma inconsistência de categorização nos dados de origem. Ainda assim, observa-se crescimento na média de Documentary em relação a 1980, e Horror permanece na última posição, com média ainda menor que a de 1980.

2. Combinação de ano/gênero com as maiores notas

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
ORDER BY avgRating DESC
LIMIT 10;

Observa-se, no entanto, um padrão recorrente de baixo volume de votos entre essas combinações, o que reduz sua representatividade estatística: notas muito altas calculadas sobre poucos votos são menos confiáveis. Refazendo a mesma ordenação, mas exigindo um mínimo de 50 mil votos, chega-se a um ranking mais robusto.

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
WHERE totalVotes > 50000
ORDER BY avgRating DESC
LIMIT 10;

3. Melhores avaliações para um gênero específico

Para o gênero Drama, buscou-se identificar os anos com as maiores médias de avaliação:

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
WHERE genre = 'Drama'
ORDER BY avgRating DESC
LIMIT 50;

O ano de 1888 aparece entre os mais bem avaliados, mas esse resultado também deve ser interpretado com cautela: está associado a uma quantidade reduzida de votos, o que compromete sua representatividade frente aos demais anos. Para o gênero Action:

In [0]:
%sql
SELECT *
FROM gold.genre_analysis
WHERE genre = 'Action'
ORDER BY avgRating DESC
LIMIT 50;

Nota-se uma tendência de avaliações mais elevadas entre os anos mais recentes para esse gênero.

4. Anos com maior número de votos

Por fim, identificaram-se os anos que concentram o maior número total de votos, somando todos os gêneros.

In [0]:
%sql
SELECT year, SUM(totalVotes) AS total_votes_per_year
FROM gold.genre_analysis
GROUP BY year
ORDER BY total_votes_per_year DESC
LIMIT 50;

Como esperado, dada a popularização da internet e a disseminação de dispositivos eletrônicos, os anos mais recentes concentram maior volume de avaliações. No entanto, o consumo de filmes não se restringe ao período de lançamento: com a consolidação das plataformas de streaming, produções mais antigas continuam sendo assistidas e avaliadas atualmente. Assim, o número de votos reflete tanto a popularidade contemporânea quanto a capacidade de um título permanecer relevante ao longo do tempo.

### Autoavaliação

De maneira geral, avalio de forma bastante positiva a experiência de desenvolvimento deste trabalho. Antes de iniciar as etapas práticas, defini como objetivo aplicar os conceitos de análise e tratamento de dados aprendidos ao longo da disciplina, construindo um pipeline de dados capaz de transformar informações brutas em conhecimento útil para responder a questões analíticas. Considero que esse objetivo foi alcançado, pois consegui realizar desde a seleção da base de dados até a criação das consultas necessárias para extrair insights relevantes.

Inicialmente, minha intenção era utilizar uma base de dados relacionada a premiações do universo musical, como o Grammy Awards, para analisar características dos álbuns premiados e identificar possíveis padrões entre gêneros musicais. Entretanto, durante a busca por conjuntos de dados no Kaggle, encontrei dificuldades para localizar uma base consistente e adequada aos objetivos propostos. Além disso, surgiram desafios relacionados à qualidade e ao tratamento dos arquivos disponíveis. Diante desse cenário, optei por utilizar a base de dados do IMDb, que apresentava maior volume de informações, melhor estruturação e possibilitava a realização de análises semelhantes às planejadas inicialmente.

Durante a execução do projeto, alguns desafios demandaram decisões importantes sobre o tratamento dos dados. Um dos primeiros foi relacionado à coluna isAdult. Inicialmente, considerei corrigir as inconsistências existentes nesse campo; porém, como essa variável não fazia parte do escopo da análise, optei por removê-la. Embora essa decisão tenha simplificado o processo, reconheço que sua exclusão pode limitar futuras análises que eventualmente utilizassem essa informação. Outro ponto de atenção ocorreu quando a chave primária tconst perdeu sua unicidade após a etapa de expansão dos gêneros. Avaliei que essa situação era esperada, uma vez que um mesmo título pode estar associado a múltiplos gêneros, e que tal característica não comprometeria os objetivos definidos para o trabalho.

Também enfrentei algumas dificuldades na elaboração de consultas e transformações de dados ao longo do projeto. Em determinados momentos, foi necessário aprofundar meus estudos e consultar materiais de apoio para compreender melhor determinadas operações e implementar as soluções desejadas. Esse processo contribuiu significativamente para o meu aprendizado, pois permitiu consolidar conhecimentos práticos sobre manipulação e análise de dados.

Ao longo das análises, identifiquei limitações que podem ter influenciado alguns resultados obtidos. Uma delas está relacionada à quantidade de avaliações disponíveis para produções mais antigas. Em diversos casos, filmes lançados nos primeiros anos da base apresentavam um número reduzido de votos, o que resultava em médias elevadas, mas pouco representativas do conjunto geral de usuários. Dessa forma, para trabalhos futuros, seria interessante estabelecer critérios mínimos de quantidade de votos ou restringir o período analisado, aumentando a confiabilidade das conclusões obtidas.

Como possibilidade de evolução do projeto, também poderiam ser incorporadas novas variáveis presentes na base do IMDb, como informações sobre profissionais envolvidos nas produções, duração dos títulos ou aspectos relacionados à popularidade. Além disso, seria possível desenvolver visualizações interativas e aplicar técnicas estatísticas mais avançadas para aprofundar a análise dos padrões identificados.

Por fim, considero que a construção do MVP foi uma experiência extremamente enriquecedora. O projeto permitiu colocar em prática os conhecimentos adquiridos ao longo da disciplina e compreender, de forma concreta, todo o processo de transformação de dados brutos em informações relevantes para a tomada de decisão. Além do desenvolvimento técnico, a atividade proporcionou uma visão mais clara do potencial da análise de dados para gerar conhecimento e apoiar a solução de problemas reais, reforçando meu interesse pela área e ampliando minha compreensão sobre a importância de um processo estruturado de tratamento e análise de dados.